# 03 - Transcribe audio (`pyneat.genai.ASRModel`)

ASR turns an audio file into text. This notebook loads the ASR model and transcribes a WAV file with
the **direct** `ASRModel` API.

> Loading the ASR model takes a minute and several GB of memory. Run these cells on the DevKit.

## Two ways to reach ASR

| Path | Command / API | Covered here |
| --- | --- | --- |
| **Direct Python API** | `pyneat.genai.ASRModel(dir).run(request)` | **this notebook** |
| **HTTP server** | POST to `/v1/audio/transcriptions` on `GenAIServer` | `05-genai-server/` |


## Memory implications

- `whisper-small-a16w8` is the ASR model used here. Its suffix `a16w8` = **16-bit activations /
  8-bit weights** (8-bit, not 4-bit like the LLM/VLM) - the trade for transcription quality is
  heavier weights per parameter.
- ASR loads an encoder + decoder; memory is dominated by the audio encoder plus the decoder KV cache.
- Transcription cost scales with **audio length**, not a token budget - a long recording takes longer
  and holds more intermediate state.

## Step 1 - Load the ASR model

In [ ]:
import pyneat as neat

MODEL_DIR = "/media/nvme/llima/models/whisper-small-a16w8"

model = neat.genai.ASRModel(MODEL_DIR)
print("model_id     :", model.model_id())
print("accepts_audio:", model.accepts_audio())   # True for an ASR model directory

**Interpretation.** `ASRModel` exposes `model_id()`, `accepts_audio()`, `run()`, and `stream()`
(header `ASRModel.h`). `accepts_audio()` must be `True`. There is no `accepts_text`/`accepts_image`
on `ASRModel` - it is audio-only; use `GenAIModel` if you want a single handle that reports all three
capabilities.

## Step 2 - Transcribe a file

Build a `GenerationRequest`, set `audio_file` to the WAV path and `language`, then `run()`. The
request's `audio_file` / `audio` / `language` fields are the ASR inputs (from `GenAITypes.h`).

In [ ]:
# Path to a local WAV/FLAC/MP3 you supply. A ready-made sample lives in the core repo at
# tests/assets/genai/audio.wav (https://github.com/sima-neat/core/blob/main/tests/assets/genai/audio.wav).
AUDIO_PATH = "assets/audio.wav"      # your audio file (relative to this notebook)

request = neat.genai.GenerationRequest()
request.audio_file = AUDIO_PATH      # path to an audio file
request.language = "en"              # default is "en"

result = model.run(request)
print("transcription:", result.text)
print("tokens:", result.metrics.generated_tokens,
      "| tok/s:", round(result.metrics.tokens_per_second, 2))

**Interpretation.** The result is the same `GenerationResult` type as the LLM/VLM path - `.text`
holds the transcript, `.metrics` holds timing. Two ways to supply audio: `request.audio_file` (a path,
used here) or `request.audio` (an audio `Tensor`, if you already have samples in memory). `language`
defaults to `"en"`; set it to the spoken language for non-English audio.

## Step 3 - Stream a transcription

`ASRModel.stream()` returns a `GenerationStream`, same as the LLM path, so a long recording can print
its transcript progressively.

In [ ]:
stream_request = neat.genai.GenerationRequest()
stream_request.audio_file = AUDIO_PATH
stream_request.language = "en"

stream_handle = model.stream(stream_request)
print("transcription: ", end="", flush=True)
for token in stream_handle:
    print(token.text, end="", flush=True)
    if token.is_final:
        break
print()

**Interpretation.** Streaming is optional for ASR but useful for long audio - you see text as
the decoder produces it instead of waiting for the whole file. Each `TokenSample.text` is the next
transcript fragment.

## References

- [`ASRModel.h`](https://github.com/sima-neat/core/blob/main/include/genai/ASRModel.h), [`GenAITypes.h`](https://github.com/sima-neat/core/blob/main/include/genai/GenAITypes.h) — `audio_file`, `audio`, `language`
- [`module.cpp`](https://github.com/sima-neat/core/blob/main/python/src/module.cpp) — `ASRModel` bindings: `run`, `stream`, `accepts_audio`
- [`genai-model.mdx`](https://github.com/sima-neat/core/blob/main/docs/develop-apps/development-workflow/genai-model.mdx) — direct ASR example
- [`request_audio_transcription.py`](https://github.com/sima-neat/core/blob/main/tutorials/021_serve_genai_models/request_audio_transcription.py) — server path